In [10]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
MODEL_NAMES = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
models = {
    name: joblib.load(ROOT / "models" / "randomized_search" / name / "unsw_nb15.joblib")
    for name in MODEL_NAMES
}
features = list(models["linear_svm"].feature_names_in_)
test_df = pd.read_parquet(
    ROOT / "datasets" / "final" / "ml" / "cicids2017_test.parquet",
    columns=features + ["id", "label_binary"],
).reset_index(drop=True)
y_test = test_df["label_binary"].eq("ATTACK").astype(int)
predictions = pd.DataFrame(
    {name: model.predict(test_df[features]) for name, model in models.items()},
    index=test_df.index,
)


In [11]:
errors = []
for name in MODEL_NAMES:
    tn, fp, fn, tp = confusion_matrix(y_test, predictions[name], labels=[0, 1]).ravel()
    errors.append({
        "modelo": name,
        "total_teste": len(y_test),
        "f1_macro": f1_score(y_test, predictions[name], average="macro"),
        "falsos_positivos": fp,
        "falsos_negativos": fn,
    })
display(pd.DataFrame(errors).sort_values("f1_macro", ascending=False))

correct = predictions.eq(y_test, axis=0)
comparisons = []
for name in MODEL_NAMES:
    if name == "linear_svm":
        continue
    for label, class_name in [(0, "BENIGN"), (1, "ATTACK")]:
        mask = y_test.eq(label)
        comparisons.append({
            "concorrente": name,
            "classe_real": class_name,
            "total_classe": int(mask.sum()),
            "svm_acerta_outro_erra": int((mask & correct["linear_svm"] & ~correct[name]).sum()),
            "svm_erra_outro_acerta": int((mask & ~correct["linear_svm"] & correct[name]).sum()),
        })
comparisons_df = pd.DataFrame(comparisons)
comparisons_df["svm_acerta_outro_erra_pct"] = 100 * comparisons_df["svm_acerta_outro_erra"] / comparisons_df["total_classe"]
comparisons_df["svm_erra_outro_acerta_pct"] = 100 * comparisons_df["svm_erra_outro_acerta"] / comparisons_df["total_classe"]
display(comparisons_df.round(2))


,modelo,total_teste,f1_macro,falsos_positivos,falsos_negativos
3,linear_svm,100925,0.699412,10089,9528
4,mlp,100925,0.631911,19636,8578
0,decision_tree,100925,0.429723,24539,17084
1,random_forest,100925,0.424111,28527,16279
2,xgboost,100925,0.402903,22773,18649


,concorrente,classe_real,total_classe,svm_acerta_outro_erra,svm_erra_outro_acerta,svm_acerta_outro_erra_pct,svm_erra_outro_acerta_pct
0,decision_tree,BENIGN,80740,17541,3091,21.73,3.83
1,decision_tree,ATTACK,20185,7721,165,38.25,0.82
2,random_forest,BENIGN,80740,21008,2570,26.02,3.18
3,random_forest,ATTACK,20185,7355,604,36.44,2.99
4,xgboost,BENIGN,80740,15181,2497,18.80,3.09
5,xgboost,ATTACK,20185,9458,337,46.86,1.67
6,mlp,BENIGN,80740,11753,2206,14.56,2.73
7,mlp,ATTACK,20185,89,1039,0.44,5.15


In [12]:
protocol_errors = []
recut_metrics = []
for target in ["unsw_nb15", "cicids2017"]:
    target_df = test_df if target == "cicids2017" else pd.read_parquet(
        ROOT / "datasets" / "final" / "ml" / f"{target}_test.parquet",
        columns=features + ["label_binary"],
    ).reset_index(drop=True)
    target_y = target_df["label_binary"].eq("ATTACK").astype(int)
    target_predictions = predictions if target == "cicids2017" else pd.DataFrame(
        {name: model.predict(target_df[features]) for name, model in models.items()},
        index=target_df.index,
    )
    retained = ~(target_df["protocol"].eq(17) & target_y.eq(1)) if target == "unsw_nb15" else target_df["protocol"].eq(6)
    for name in MODEL_NAMES:
        recut_metrics.append({
            "base_teste": target, "modelo": name,
            "recorte": "Sem ataques UDP" if target == "unsw_nb15" else "Somente TCP",
            "f1_original": f1_score(target_y, target_predictions[name], average="macro"),
            "f1_recorte": f1_score(target_y.loc[retained], target_predictions.loc[retained, name], average="macro"),
        })
    for protocol, group in target_df.groupby("protocol"):
        actual = target_y.loc[group.index]
        for name in MODEL_NAMES:
            tn, fp, fn, tp = confusion_matrix(
                actual, target_predictions.loc[group.index, name], labels=[0, 1]
            ).ravel()
            protocol_errors.append({
                "base_teste": target, "protocolo": protocol, "modelo": name,
                "benignos": tn + fp, "ataques": fn + tp,
                "falsos_positivos": fp, "falsos_negativos": fn,
                "taxa_falsos_positivos": fp / (tn + fp) if tn + fp else float("nan"),
                "recall_ataques": tp / (fn + tp) if fn + tp else float("nan"),
                "recall_benignos": tn / (tn + fp) if tn + fp else float("nan"),
            })

protocol_errors_df = pd.DataFrame(protocol_errors)
display(protocol_errors_df.round(4))
svm_fn = (
    protocol_errors_df.query("base_teste == 'unsw_nb15' and modelo == 'linear_svm'")
    .set_index("protocolo")["falsos_negativos"]
)
print(f"Falsos negativos do SVM na UNSW que são UDP: {svm_fn.loc[17]} de {svm_fn.sum()} ({svm_fn.loc[17] / svm_fn.sum():.2%})")

udp_unsw = protocol_errors_df.query("base_teste == 'unsw_nb15' and protocolo == 17").sort_values("recall_ataques")
udp_table = udp_unsw[["modelo", "falsos_negativos", "recall_ataques", "recall_benignos"]].copy()
udp_table.columns = ["Modelo", "Falsos negativos UDP", "Recall ataques UDP", "Recall benignos UDP"]
display(
    udp_table.style
    .set_caption(f"UNSW-NB15: {udp_unsw.ataques.iloc[0]} ataques UDP e {udp_unsw.benignos.iloc[0]} UDP benignos")
    .format({"Falsos negativos UDP": "{:,.0f}", "Recall ataques UDP": "{:.2%}", "Recall benignos UDP": "{:.2%}"}, decimal=",", thousands=".")
    .hide(axis="index")
)

recut_df = pd.DataFrame(recut_metrics)
display(recut_df.round(4))


,base_teste,protocolo,modelo,benignos,ataques,falsos_positivos,falsos_negativos,taxa_falsos_positivos,recall_ataques,recall_benignos
0,unsw_nb15,6,decision_tree,57966,16081,943,67,0.0163,0.9958,0.9837
1,unsw_nb15,6,random_forest,57966,16081,953,11,0.0164,0.9993,0.9836
2,unsw_nb15,6,xgboost,57966,16081,946,20,0.0163,0.9988,0.9837
3,unsw_nb15,6,linear_svm,57966,16081,1002,171,0.0173,0.9894,0.9827
4,unsw_nb15,6,mlp,57966,16081,971,15,0.0168,0.9991,0.9832
5,unsw_nb15,17,decision_tree,22102,4103,289,33,0.0131,0.9920,0.9869
6,unsw_nb15,17,random_forest,22102,4103,292,19,0.0132,0.9954,0.9868
7,unsw_nb15,17,xgboost,22102,4103,301,7,0.0136,0.9983,0.9864
8,unsw_nb15,17,linear_svm,22102,4103,172,2661,0.0078,0.3515,0.9922
9,unsw_nb15,17,mlp,22102,4103,323,38,0.0146,0.9907,0.9854


Falsos negativos do SVM na UNSW que são UDP: 2661 de 2832 (93.96%)


Modelo,Falsos negativos UDP,Recall ataques UDP,Recall benignos UDP
linear_svm,2.661,"35,15%","99,22%"
mlp,38,"99,07%","98,54%"
decision_tree,33,"99,20%","98,69%"
random_forest,19,"99,54%","98,68%"
xgboost,7,"99,83%","98,64%"


,base_teste,modelo,recorte,f1_original,f1_recorte
0,unsw_nb15,decision_tree,Sem ataques UDP,0.9798,0.9765
1,unsw_nb15,random_forest,Sem ataques UDP,0.9807,0.9773
2,unsw_nb15,xgboost,Sem ataques UDP,0.9807,0.9771
3,unsw_nb15,linear_svm,Sem ataques UDP,0.9360,0.9755
4,unsw_nb15,mlp,Sem ataques UDP,0.9796,0.9763
5,cicids2017,decision_tree,Somente TCP,0.4297,0.4526
6,cicids2017,random_forest,Somente TCP,0.4241,0.4559
7,cicids2017,xgboost,Somente TCP,0.4029,0.4080
8,cicids2017,linear_svm,Somente TCP,0.6994,0.6687
9,cicids2017,mlp,Somente TCP,0.6319,0.6440
